# fiftyone_explore.ipynb — browse a RAW per-class export

**When to use this:** *after* a raw acquisition pull, *before* Stage 5.2 conversion. Loads an already-acquired raw per-class export straight from disk — no network calls, safe to re-run anytime.

**What it expects:** the raw COCO-style layout `acquire_openimages.py` produces — `dataset/raw/open_images/<class>/` containing a `data/` image folder + `labels.json`, native (non-canonical) class names, unfiltered structure.

**Not for:** converted/processed data (`dataset/processed/<source>/`, flat `images/`+`labels/`, canonical class ids). That's a different schema (DEC-046) — use `fiftyone_review_processed.ipynb` for that instead. Pointing this notebook at a processed directory will fail with a `Data directory '.../data/' does not exist` error, since processed output has no `data/` subfolder.

In [ ]:
# Imports
import sys
from pathlib import Path

import fiftyone as fo
from fiftyone import ViewField as F


def _find_repo_root(start: Path) -> Path:
    """Walk up from `start` to find the repo root (has config/ + AGENTS.md).

    Needed because notebooks live in notebooks/, not the repo root, and
    Jupyter's working directory depends on how it was launched — this
    makes every path below robust regardless of that.
    """
    for parent in [start, *start.parents]:
        if (parent / "config").is_dir() and (parent / "AGENTS.md").is_file():
            return parent
    raise RuntimeError("Could not locate repo root from notebook cwd.")


REPO_ROOT = _find_repo_root(Path.cwd())
sys.path.insert(0, str(REPO_ROOT))

In [ ]:
# Get directory of specific class

export_dir = REPO_ROOT / "dataset/raw/open_images/chairs"

In [ ]:
# Load the already-exported COCO directory back into FiftyOne.
# No network calls here — the images are already local, so this is
# fast and safe to re-run as often as you want.
dataset_name = f"explore_{export_dir.name}"

dataset = fo.Dataset.from_dir(
    dataset_dir=str(export_dir),
    dataset_type=fo.types.COCODetectionDataset,
    name=dataset_name,
    persistent=False,
    overwrite=True,
)
print(dataset)

In [ ]:
# Launch the App
session = fo.launch_app(dataset, auto=False)

In [ ]:
session